## Cell 1 — Installing libraries

In [ ]:
!pip install transformers torch langdetect sentencepiece -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 33.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


## Cell 2 — Loading the models

In [ ]:
from transformers import MarianMTModel, MarianTokenizer

print("Loading Arabic → English model...")
ar_en_model_name = "Helsinki-NLP/opus-mt-ar-en"
ar_en_tokenizer = MarianTokenizer.from_pretrained(ar_en_model_name)
ar_en_model = MarianMTModel.from_pretrained(ar_en_model_name)

print("Loading English → Arabic model...")
en_ar_model_name = "Helsinki-NLP/opus-mt-en-ar"
en_ar_tokenizer = MarianTokenizer.from_pretrained(en_ar_model_name)
en_ar_model = MarianMTModel.from_pretrained(en_ar_model_name)

print("Both models loaded!")

Loading Arabic → English model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/917k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

Both models loaded!


## Cell 3 — The translation function

In [ ]:
from langdetect import detect

def translate(text):
    try:
        lang = detect(text)
    except:
        lang = "en"

    if lang == "ar":
        # Arabic → English
        inputs = ar_en_tokenizer(text, return_tensors="pt", padding=True)
        outputs = ar_en_model.generate(**inputs)
        translated = ar_en_tokenizer.decode(outputs[0], skip_special_tokens=True)
        direction = "Arabic → English"
    else:
        # English → Arabic
        inputs = en_ar_tokenizer(text, return_tensors="pt", padding=True)
        outputs = en_ar_model.generate(**inputs)
        translated = en_ar_tokenizer.decode(outputs[0], skip_special_tokens=True)
        direction = "English → Arabic"

    return {
        "original": text,
        "translated": translated,
        "direction": direction,
        "detected_language": lang
    }

print("Function ready!")

Function ready!


## Cell 4 — Testing

In [ ]:
# Test English → Arabic
test1 = translate("Hello, how are you?")
print(f"Direction: {test1['direction']}")
print(f"Original:  {test1['original']}")
print(f"Translated: {test1['translated']}")
print()

# Test Arabic → English
test2 = translate("مرحبا، كيف حالك؟")
print(f"Direction: {test2['direction']}")
print(f"Original:  {test2['original']}")
print(f"Translated: {test2['translated']}")
print()

# Test more phrases
phrases = [
    "Good morning, welcome to Dubai!",
    "أنا سعيد جداً بلقائك",
    "The weather is beautiful today",
    "شكراً جزيلاً على مساعدتك"
]

print("--- Batch Test ---")
for phrase in phrases:
    result = translate(phrase)
    print(f"{result['direction']}: {result['original']}")
    print(f"→ {result['translated']}")
    print()

Direction: English → Arabic
Original:  Hello, how are you?
Translated: مرحباً، كيف حالك؟

Direction: Arabic → English
Original:  مرحبا، كيف حالك؟
Translated: Hey, how are you?

--- Batch Test ---
English → Arabic: Good morning, welcome to Dubai!
→ صباح الخير، مرحبا بكم في دبي!

Arabic → English: أنا سعيد جداً بلقائك
→ I'm so happy to meet you.

English → Arabic: The weather is beautiful today
→ الطقس جميل اليوم

Arabic → English: شكراً جزيلاً على مساعدتك
→ Thank you so much for your help.



## Cell 5 — Load sentiment models:

In [ ]:
# Day 2 - Sentiment Detection
!pip install transformers torch -q
# Already installed but just making sure
print("Ready for sentiment models!")

Ready for sentiment models!


## Cell 6 — Sentiment function:

In [ ]:
from transformers import pipeline

print("Loading English sentiment model...")
english_sentiment = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

print("Loading Arabic sentiment model...")
arabic_sentiment = pipeline(
    "sentiment-analysis",
    model="CAMeL-Lab/bert-base-arabic-camelbert-da-sentiment"
)

print("Both sentiment models loaded!")

Loading English sentiment model...


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/86.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Both sentiment models loaded!


## Cell 7 — Test sentiment:

In [ ]:
def get_sentiment(text):
    try:
        lang = detect(text)
    except:
        lang = "en"

    if lang == "ar":
        result = arabic_sentiment(text)[0]
    else:
        result = english_sentiment(text)[0]

    # Normalise labels to consistent format
    label = result["label"].upper()
    if label in ["LABEL_0", "NEGATIVE", "NEG"]:
        label = "NEGATIVE 😞"
        emoji = "😞"
    elif label in ["LABEL_1", "POSITIVE", "POS"]:
        label = "POSITIVE 😊"
        emoji = "😊"
    else:
        label = "NEUTRAL 😐"
        emoji = "😐"

    confidence = round(result["score"] * 100, 1)

    return {
        "text": text,
        "language": lang,
        "sentiment": label,
        "confidence": confidence
    }

print("Sentiment function ready!")

Sentiment function ready!


## Cell 8 — Combine translation + sentiment together:

In [ ]:
test_phrases = [
    "I love Dubai so much, it's amazing!",
    "This service is terrible, I'm very angry",
    "The weather is okay today",
    "أنا سعيد جداً بهذا المكان",
    "هذا الطعام سيء جداً",
    "الجو معتدل اليوم"
]

print("--- Sentiment Analysis Test ---\n")
for phrase in test_phrases:
    result = get_sentiment(phrase)
    lang_label = "🇬🇧 English" if result["language"] == "en" else "🇦🇪 Arabic"
    print(f"{lang_label}: {result['text']}")
    print(f"Sentiment: {result['sentiment']} | Confidence: {result['confidence']}%")
    print()

--- Sentiment Analysis Test ---

🇬🇧 English: I love Dubai so much, it's amazing!
Sentiment: POSITIVE 😊 | Confidence: 100.0%

🇬🇧 English: This service is terrible, I'm very angry
Sentiment: NEGATIVE 😞 | Confidence: 100.0%

🇬🇧 English: The weather is okay today
Sentiment: POSITIVE 😊 | Confidence: 100.0%

🇦🇪 Arabic: أنا سعيد جداً بهذا المكان
Sentiment: POSITIVE 😊 | Confidence: 98.6%

🇦🇪 Arabic: هذا الطعام سيء جداً
Sentiment: NEGATIVE 😞 | Confidence: 99.4%

🇦🇪 Arabic: الجو معتدل اليوم
Sentiment: NEGATIVE 😞 | Confidence: 63.3%



## Cell 9 — Final combined test:

In [ ]:
test_inputs = [
    "I'm really excited to visit Dubai next week!",
    "أنا غاضب جداً من هذه الخدمة السيئة",
    "The meeting was cancelled unfortunately",
    "الحمد لله، كل شيء على ما يرام"
]

print("=== BRIDGEAI FULL PIPELINE TEST ===\n")
for text in test_inputs:
    result = analyze(text)
    print(f"INPUT:       {result['original']}")
    print(f"TRANSLATION: {result['translated']}")
    print(f"DIRECTION:   {result['direction']}")
    print(f"SENTIMENT (original):    {result['sentiment_original']} ({result['sentiment_original_confidence']}%)")
    print(f"SENTIMENT (translated):  {result['sentiment_translated']} ({result['sentiment_translated_confidence']}%)")
    print("-" * 60)

=== BRIDGEAI FULL PIPELINE TEST ===

INPUT:       I'm really excited to visit Dubai next week!
TRANSLATION: أنا متحمسة حقاً لزيارة (دبي) الأسبوع القادم!
DIRECTION:   English → Arabic
SENTIMENT (original):    POSITIVE 😊 (100.0%)
SENTIMENT (translated):  POSITIVE 😊 (96.7%)
------------------------------------------------------------
INPUT:       أنا غاضب جداً من هذه الخدمة السيئة
TRANSLATION: I'm so mad at this shitty service.
DIRECTION:   Arabic → English
SENTIMENT (original):    NEGATIVE 😞 (98.7%)
SENTIMENT (translated):  NEGATIVE 😞 (100.0%)
------------------------------------------------------------
INPUT:       The meeting was cancelled unfortunately
TRANSLATION: تم إلغاء الجلسة لسوء الحظ
DIRECTION:   English → Arabic
SENTIMENT (original):    NEGATIVE 😞 (100.0%)
SENTIMENT (translated):  NEGATIVE 😞 (91.5%)
------------------------------------------------------------
INPUT:       الحمد لله، كل شيء على ما يرام
TRANSLATION: Thank God. Everything's fine.
DIRECTION:   Arabic → English
SEN

In [ ]:
def analyze(text):
    try:
        lang = detect(text)
    except:
        lang = "en"

    translation = translate(text)
    sentiment_original = get_sentiment(text)
    sentiment_translated = get_sentiment(translation["translated"])

    return {
        "original": text,
        "translated": translation["translated"],
        "direction": translation["direction"],
        "sentiment_original": sentiment_original["sentiment"],
        "sentiment_original_confidence": sentiment_original["confidence"],
        "sentiment_translated": sentiment_translated["sentiment"],
        "sentiment_translated_confidence": sentiment_translated["confidence"],
    }

# Final test
test_inputs = [
    "I'm really excited to visit Dubai next week!",
    "أنا غاضب جداً من هذه الخدمة السيئة",
    "الحمد لله، كل شيء على ما يرام"
]

print("=== BRIDGEAI FULL PIPELINE ===\n")
for text in test_inputs:
    result = analyze(text)
    print(f"INPUT:       {result['original']}")
    print(f"TRANSLATION: {result['translated']}")
    print(f"DIRECTION:   {result['direction']}")
    print(f"SENTIMENT (original):   {result['sentiment_original']} ({result['sentiment_original_confidence']}%)")
    print(f"SENTIMENT (translated): {result['sentiment_translated']} ({result['sentiment_translated_confidence']}%)")
    print("-" * 60)

=== BRIDGEAI FULL PIPELINE ===

INPUT:       I'm really excited to visit Dubai next week!
TRANSLATION: أنا متحمسة حقاً لزيارة (دبي) الأسبوع القادم!
DIRECTION:   English → Arabic
SENTIMENT (original):   POSITIVE 😊 (100.0%)
SENTIMENT (translated): POSITIVE 😊 (96.7%)
------------------------------------------------------------
INPUT:       أنا غاضب جداً من هذه الخدمة السيئة
TRANSLATION: I'm so mad at this shitty service.
DIRECTION:   Arabic → English
SENTIMENT (original):   NEGATIVE 😞 (98.7%)
SENTIMENT (translated): NEGATIVE 😞 (100.0%)
------------------------------------------------------------
INPUT:       الحمد لله، كل شيء على ما يرام
TRANSLATION: Thank God. Everything's fine.
DIRECTION:   Arabic → English
SENTIMENT (original):   POSITIVE 😊 (97.0%)
SENTIMENT (translated): POSITIVE 😊 (100.0%)
------------------------------------------------------------
